# This model is trained on (2020/01/01  to  2023/12/30) 48 month data with STATIC MASK

In [6]:
# import xarray as xr
# import numpy as np

# # --- Corrected File Paths ---
# PATH_CMEMS = r'dAtA/cmems_mod_ibi_wav_my_0.027deg_static_1756186965833.nc'
# PATH_GEBCO = r"C:\Users\user\Documents\GitHub\Wave-Prediction\dAtA\Sea_surface\gebco_2022_n40.5_s38.5_w-11.0_e-8.5.nc"
# PATH_OUTPUT_NEW_STATIC = r'dAtA/cmems_mod_ibi_wav_my_0.027deg_static_GEBCO_resampled.nc'

# # --- STEP 1: Inspecting the Datasets ---
# try:
#     print("--- 1. CMEMS Dataset Details ---")
#     with xr.open_dataset(PATH_CMEMS) as ds_cmems:
#         print(ds_cmems)

#     print("\n" + "="*50 + "\n")

#     print("--- 2. GEBCO Dataset Details ---")
#     with xr.open_dataset(PATH_GEBCO) as ds_gebco:
#         print(ds_gebco)

#     print("\n" + "="*50 + "\n")

# # --- STEP 2: Resampling and Saving ---
#     # Load datasets again for processing
#     ds_cmems = xr.open_dataset(PATH_CMEMS)
#     ds_gebco = xr.open_dataset(PATH_GEBCO)

#     # Standardize Coordinate Names to prevent errors
#     ds_gebco = ds_gebco.rename({'lat': 'latitude', 'lon': 'longitude'})
#     print("✅ GEBCO coordinates renamed for compatibility.")

#     # Interpolate the high-resolution data onto the model's grid
#     ds_gebco_resampled = ds_gebco.interp_like(ds_cmems)
#     print("✅ GEBCO data successfully resampled to match CMEMS grid.")

#     # Prepare data variables
#     new_depth_data = ds_gebco_resampled['elevation'].values * -1
#     original_mask = ds_cmems['mask'].values

#     # Create the new, combined dataset
#     ds_new_static = xr.Dataset(
#         data_vars={
#             'deptho': (('latitude', 'longitude'), new_depth_data),
#             'mask': (('latitude', 'longitude'), original_mask)
#         },
#         coords={
#             'latitude': ds_cmems['latitude'],
#             'longitude': ds_cmems['longitude']
#         }
#     )

#     # Save the final file
#     ds_new_static.to_netcdf(PATH_OUTPUT_NEW_STATIC)
#     print(f"\n✅ Successfully created new static file at: {PATH_OUTPUT_NEW_STATIC}")

#     # Verification
#     print("\n--- Verifying Created File ---")
#     with xr.open_dataset(PATH_OUTPUT_NEW_STATIC) as ds_verify:
#         print(ds_verify)

# except FileNotFoundError as e:
#     print(f"❌ ERROR: A file was not found. Please double-check the corrected paths.")
#     print(e)
# except Exception as e:
#     print(f"An error occurred: {e}")

In [7]:
import torch

# Check if CUDA is available
cuda_available = torch.cuda.is_available()
print(f"Is CUDA available? {cuda_available}")

# If it is available, check how many GPUs are detected
if cuda_available:
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Name: {torch.cuda.get_device_name(0)}")

# Check the CUDA version PyTorch was built with
print(f"PyTorch CUDA Version: {torch.version.cuda}")

Is CUDA available? True
Number of GPUs: 2
Current GPU Name: NVIDIA GeForce GTX 1080 Ti
PyTorch CUDA Version: 12.1


In [11]:
# ==============================================================================
# MASTER CONFIGURATION CELL
# ==============================================================================
import torch
import os

# --- Core Parameters ---
# Change this value to 24, 36, 72, etc., to test different models
LOOKBACK_HOURS = 48
# Forecast period is now 7 days (7 * 24 = 168)
FORECAST_HORIZON_HOURS = 168

# --- File Paths ---
# MODIFICATION: Updated to the new file paths you provided
RAW_DATA_PATH = r'c:\Users\user\Documents\GitHub\Wave-Prediction\dAtA\cmems_mod_ibi_wav_my_0.027deg_PT1H-i_multi-vars_11.00W-8.53W_38.50N-40.47N_2020-01-01-2023-12-30.nc'
STATIC_DATA_PATH = r'C:\Users\user\Documents\GitHub\Wave-Prediction\dAtA\Sea_surface\cmems_mod_ibi_wav_my_0.027deg_static_GEBCO_resampled.nc'

# MODIFICATION: File paths are now dynamic to keep experiments separate
BASE_DIR = r'D:\babe_prediction'
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, f'processed_data_lookback_{LOOKBACK_HOURS}_static')
MODEL_SAVE_PATH = f'convlstm_lookback_{LOOKBACK_HOURS}_forecast_{FORECAST_HORIZON_HOURS}_static.pth'

# --- Feature Engineering ---
# These are the TIME-VARYING features
VARS_TO_USE = ['VCMX','VSDmag', 'VTM10', 'VTM02', 'VTM01_WW', 'VTM01_SW1', 'VMXL', 'VHM0_WW', 'VHM0_SW1']
TARGET_VAR = 'VCMX'
# MODIFICATION: Total input channels = 9 time-varying + 2 static (depth, mask)
INPUT_CHANNELS = len(VARS_TO_USE) + 2

# --- Training Hyperparameters ---
LEARNING_RATE = 1e-5
BATCH_SIZE = 4
EPOCHS = 50
# MODIFICATION: Early stopping patience is now a configurable parameter
EARLY_STOPPING_PATIENCE = 5

# --- System Configuration ---
NUM_WORKERS = 0  # Set to 0 if you encounter multiprocessing issues, 4+ for speed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Print a summary of the current configuration ---
print("--- Configuration Summary ---")
print(f"Lookback Period: {LOOKBACK_HOURS} hours")
print(f"Forecast Horizon: {FORECAST_HORIZON_HOURS} hours")
print(f"Input Channels: {INPUT_CHANNELS} ({len(VARS_TO_USE)} time-varying + 2 static)")
print(f"Processed Data Path: {PROCESSED_DATA_DIR}")
print(f"Model Save Path: {MODEL_SAVE_PATH}")
print(f"Using Device: {device}")
print("---------------------------")

--- Configuration Summary ---
Lookback Period: 48 hours
Forecast Horizon: 168 hours
Input Channels: 11 (9 time-varying + 2 static)
Processed Data Path: D:\babe_prediction\processed_data_lookback_48_static
Model Save Path: convlstm_lookback_48_forecast_168_static.pth
Using Device: cuda
---------------------------


# STEP 1: PRE-PROCESSING SCRIPT (with Static Features)

In [12]:
# ==============================================================================
# ==============================================================================
import os
import pickle
import xarray as xr
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler
from tqdm.auto import tqdm
import warnings
import shutil

warnings.filterwarnings('ignore')

if os.path.exists(PROCESSED_DATA_DIR):
    print(f"Removing old processed data directory: {PROCESSED_DATA_DIR}")
    shutil.rmtree(PROCESSED_DATA_DIR)

print("\n--- Starting Full Pre-processing Workflow ---")

# ==============================================================================
# Step 1: Load and Clean Data (Time-Varying and Static)
# ==============================================================================
print("\n[Step 1/5] Loading and cleaning all data...")

# Load Time-Varying Data
try:
    ds_raw = xr.open_dataset(RAW_DATA_PATH)
except FileNotFoundError:
    raise SystemExit(f"❌ ERROR: Raw data file not found at {RAW_DATA_PATH}")

ds_raw['VSDmag'] = np.sqrt(ds_raw['VSDX']**2 + ds_raw['VSDY']**2)
ds_clean = ds_raw[VARS_TO_USE].astype(np.float32).fillna(0)
print("✅ Time-varying data loaded and cleaned.")

# MODIFICATION: Load Static Data from the .nc file
try:
    ds_static = xr.open_dataset(STATIC_DATA_PATH)
    # The new file is already cleaned and has the correct variable names
    ocean_depth = ds_static['deptho'].values.astype(np.float32)
    ocean_mask = ds_static['mask'].values.astype(np.float32)
except FileNotFoundError:
    raise SystemExit(f"❌ ERROR: Static data file not found at {STATIC_DATA_PATH}.")
print("✅ Static data loaded and cleaned from new GEBCO file.")

# ==============================================================================
# Step 2: Define Data Splits
# ==============================================================================
print("\n[Step 2/5] Splitting data into train, validation, and test sets...")
ds_train = ds_clean.sel(time=slice('2020-01-01', '2022-12-31'))
ds_val = ds_clean.sel(time=slice('2023-01-01', '2023-06-30'))
ds_test = ds_clean.sel(time=slice('2023-07-01', '2023-12-30'))
ds_splits = {'train': ds_train, 'val': ds_val, 'test': ds_test}
print(f"Train split: {len(ds_train.time)} time steps")
print(f"Validation split: {len(ds_val.time)} time steps")
print(f"Test split: {len(ds_test.time)} time steps")

# ==============================================================================
# Step 3: Create and Save Scalers (Including for Static Features)
# ==============================================================================
print("\n[Step 3/5] Fitting scalers on TRAINING data only...")
scalers = {}
for var in tqdm(VARS_TO_USE, desc="Fitting Time-Varying Scalers"):
    data_to_fit = ds_train[var].values.reshape(-1, 1)
    scaler = MinMaxScaler()
    scaler.fit(data_to_fit)
    scalers[var] = scaler

print("Fitting static feature scalers...")
depth_scaler = MinMaxScaler()
depth_scaler.fit(ocean_depth.reshape(-1, 1))
scalers['ocean_depth'] = depth_scaler

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
scaler_path = os.path.join(PROCESSED_DATA_DIR, 'scalers.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scalers, f)
print(f"✅ All scalers fitted and saved to '{scaler_path}'")

# ==============================================================================
# Step 4: Scale Static Features
# ==============================================================================
print("\n[Step 4/5] Scaling static features...")
scaled_depth = scalers['ocean_depth'].transform(ocean_depth.reshape(-1, 1)).reshape(ocean_depth.shape)
scaled_depth_ch = np.expand_dims(scaled_depth, axis=0)
ocean_mask_ch = np.expand_dims(ocean_mask, axis=0)
static_features_np = np.concatenate([scaled_depth_ch, ocean_mask_ch], axis=0)
print("✅ Static features scaled.")

# ==============================================================================
# Step 5: Process and Save Samples (Now with Static Features)
# ==============================================================================
print("\n[Step 5/5] Generating and saving individual samples...")
total_window_size = LOOKBACK_HOURS + FORECAST_HORIZON_HOURS

for split_name, ds_split in ds_splits.items():
    print(f"\n--- Processing '{split_name}' split ---")
    split_dir = os.path.join(PROCESSED_DATA_DIR, split_name)
    os.makedirs(split_dir, exist_ok=True)
    
    num_sequences = len(ds_split['time']) - total_window_size
    if num_sequences < 0:
        print(f"⚠️ Warning: '{split_name}' split is too small. Skipping.")
        continue
        
    for i in tqdm(range(num_sequences), desc=f"Saving {split_name} samples"):
        window_slice = ds_split.isel(time=slice(i, i + total_window_size))
        
        scaled_window_vars = []
        for var in VARS_TO_USE:
            data = window_slice[var].values
            scaled_data = scalers[var].transform(data.reshape(-1, 1)).reshape(data.shape)
            scaled_window_vars.append(scaled_data)
        
        scaled_window_tensor = np.stack(scaled_window_vars, axis=1)

        X_tv_np = scaled_window_tensor[:LOOKBACK_HOURS, :, :, :]
        y_np = scaled_window_tensor[LOOKBACK_HOURS:, VARS_TO_USE.index(TARGET_VAR), :, :]

        static_features_tiled = np.tile(static_features_np, (LOOKBACK_HOURS, 1, 1, 1))
        X_final_np = np.concatenate([X_tv_np, static_features_tiled], axis=1)

        X = torch.from_numpy(X_final_np.astype(np.float32))
        y = torch.from_numpy(y_np.astype(np.float32))
        
        sample_path = os.path.join(split_dir, f'sample_{i:06d}.pt')
        torch.save((X, y), sample_path)

print("\n\n✅ Pre-processing complete with static features!")
print(f"Clean, processed data is now available at: '{PROCESSED_DATA_DIR}'")


--- Starting Full Pre-processing Workflow ---

[Step 1/5] Loading and cleaning all data...
✅ Time-varying data loaded and cleaned.
✅ Static data loaded and cleaned from new GEBCO file.

[Step 2/5] Splitting data into train, validation, and test sets...
Train split: 26304 time steps
Validation split: 4344 time steps
Test split: 4392 time steps

[Step 3/5] Fitting scalers on TRAINING data only...


Fitting Time-Varying Scalers:   0%|          | 0/9 [00:00<?, ?it/s]

Fitting static feature scalers...
✅ All scalers fitted and saved to 'D:\babe_prediction\processed_data_lookback_48_static\scalers.pkl'

[Step 4/5] Scaling static features...
✅ Static features scaled.

[Step 5/5] Generating and saving individual samples...

--- Processing 'train' split ---


Saving train samples:   0%|          | 0/26088 [00:00<?, ?it/s]


--- Processing 'val' split ---


Saving val samples:   0%|          | 0/4128 [00:00<?, ?it/s]


--- Processing 'test' split ---


Saving test samples:   0%|          | 0/4176 [00:00<?, ?it/s]



✅ Pre-processing complete with static features!
Clean, processed data is now available at: 'D:\babe_prediction\processed_data_lookback_48_static'


# STEP 2: TRAINING SCRIPT

In [13]:
# ==============================================================================
# ==============================================================================
import os
import pickle
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import time
import numpy as np

# --- Dataset and Model Classes ---
class PreprocessedWaveDataset(Dataset):
    def __init__(self, split_dir):
        self.file_paths = sorted(glob.glob(os.path.join(split_dir, '*.pt')))
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        return torch.load(self.file_paths[idx])

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__(); self.input_dim, self.hidden_dim, self.kernel_size, self.bias = input_dim, hidden_dim, kernel_size, bias; self.padding = kernel_size[0] // 2; self.conv = nn.Conv2d(self.input_dim + self.hidden_dim, 4 * self.hidden_dim, self.kernel_size, padding=self.padding, bias=self.bias)
    def forward(self, x, h_c): h, c = h_c; combined = torch.cat([x, h], dim=1); cc = self.conv(combined); i, f, o, g = torch.split(cc, self.hidden_dim, dim=1); i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g); c_n = f * c + i * g; h_n = o * torch.tanh(c_n); return h_n, c_n
    def init_hidden(self, b, i, d): h, w = i; return (torch.zeros(b, self.hidden_dim, h, w, device=d), torch.zeros(b, self.hidden_dim, h, w, device=d))

class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, batch_first=True, bias=True):
        super(ConvLSTM, self).__init__(); self.batch_first, self.num_layers = batch_first, num_layers; hidden_dims = [hidden_dim] * num_layers if isinstance(hidden_dim, int) else hidden_dim; cell_list = [];
        for i in range(self.num_layers): cur_input_dim = input_dim if i == 0 else hidden_dims[i - 1]; cell_list.append(ConvLSTMCell(cur_input_dim, hidden_dims[i], kernel_size, bias)); self.cell_list = nn.ModuleList(cell_list)
    def forward(self, x, h_c=None):
        b, s_l, _, h, w = x.size();
        if h_c is None: h_c = self._init_hidden(b, (h, w), x.device)
        cur_in = x
        for l_idx in range(self.num_layers):
            h, c = h_c[l_idx]; output_inner = []
            for t in range(s_l): h, c = self.cell_list[l_idx](cur_in[:, t, :, :, :], [h, c]); output_inner.append(h)
            cur_in = torch.stack(output_inner, dim=1)
        return cur_in, [h, c]
    def _init_hidden(self, b, i, d): return [cell.init_hidden(b, i, d) for cell in self.cell_list]

class ConvLSTMNet(nn.Module):
    def __init__(self, input_dim, forecast_horizon, hidden_dims=[64, 32], kernel_size=(3, 3)):
        super(ConvLSTMNet, self).__init__(); self.cl1 = ConvLSTM(input_dim, hidden_dims[0], kernel_size, 1, batch_first=True); self.cl2 = ConvLSTM(hidden_dims[0], hidden_dims[1], kernel_size, 1, batch_first=True); self.output_conv = nn.Conv2d(hidden_dims[1], forecast_horizon, kernel_size=(1, 1), padding='same')
    def forward(self, x_seq): l1_o, _ = self.cl1(x_seq); l2_o, _ = self.cl2(l1_o); return self.output_conv(l2_o[:, -1, :, :, :])

def train_model(model, train_loader, val_loader, device, epochs, lr, patience, model_path):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.MSELoss(); best_val_loss = float('inf'); scaler = torch.cuda.amp.GradScaler(); epochs_no_improve = 0
    print("\n--- Starting Model Training ---")
    for epoch in range(epochs):
        start_time = time.time(); model.train(); total_train_loss = 0.0; train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]")
        for X, y in train_pbar:
            X, y = X.to(device), y.to(device); optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(): predicted = model(X); target = y; loss = loss_fn(predicted, target)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); total_train_loss += loss.item(); train_pbar.set_postfix({'loss': f'{loss.item():.6f}'})
        avg_train_loss = total_train_loss / len(train_loader); model.eval(); total_val_loss = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                with torch.cuda.amp.autocast(): predicted = model(X); target = y; loss = loss_fn(predicted, target)
                total_val_loss += loss.item()
        avg_val_loss = total_val_loss / len(val_loader); epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}")
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss; epochs_no_improve = 0; torch.save(model.state_dict(), model_path); print(f"✅ New best model saved with validation loss: {best_val_loss:.6f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience: print(f"Early stopping triggered after {epoch+1} epochs."); break
    print(f"\n✅ Training completed. Best model saved to {model_path}"); model.load_state_dict(torch.load(model_path)); return model

In [2]:
# # ==============================================================================
# # ==============================================================================
# import os
# import pickle
# import glob
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# from tqdm.auto import tqdm
# import time
# import numpy as np

# # --- Dataset and Model Classes ---
# class PreprocessedWaveDataset(Dataset):
#     def __init__(self, split_dir):
#         self.file_paths = sorted(glob.glob(os.path.join(split_dir, '*.pt')))
#     def __len__(self):
#         return len(self.file_paths)
#     def __getitem__(self, idx):
#         return torch.load(self.file_paths[idx])

# class ConvLSTMCell(nn.Module):
#     def __init__(self, input_dim, hidden_dim, kernel_size, bias):
#         super(ConvLSTMCell, self).__init__(); self.input_dim, self.hidden_dim, self.kernel_size, self.bias = input_dim, hidden_dim, kernel_size, bias; self.padding = kernel_size[0] // 2; self.conv = nn.Conv2d(self.input_dim + self.hidden_dim, 4 * self.hidden_dim, self.kernel_size, padding=self.padding, bias=self.bias)
#     def forward(self, x, h_c): h, c = h_c; combined = torch.cat([x, h], dim=1); cc = self.conv(combined); i, f, o, g = torch.split(cc, self.hidden_dim, dim=1); i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g); c_n = f * c + i * g; h_n = o * torch.tanh(c_n); return h_n, c_n
#     def init_hidden(self, b, i, d): h, w = i; return (torch.zeros(b, self.hidden_dim, h, w, device=d), torch.zeros(b, self.hidden_dim, h, w, device=d))

# class ConvLSTM(nn.Module):
#     def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, batch_first=True, bias=True):
#         super(ConvLSTM, self).__init__(); self.batch_first, self.num_layers = batch_first, num_layers; hidden_dims = [hidden_dim] * num_layers if isinstance(hidden_dim, int) else hidden_dim; cell_list = [];
#         for i in range(self.num_layers): cur_input_dim = input_dim if i == 0 else hidden_dims[i - 1]; cell_list.append(ConvLSTMCell(cur_input_dim, hidden_dims[i], kernel_size, bias)); self.cell_list = nn.ModuleList(cell_list)
#     def forward(self, x, h_c=None):
#         b, s_l, _, h, w = x.size();
#         if h_c is None: h_c = self._init_hidden(b, (h, w), x.device)
#         cur_in = x
#         for l_idx in range(self.num_layers):
#             h, c = h_c[l_idx]; output_inner = []
#             for t in range(s_l): h, c = self.cell_list[l_idx](cur_in[:, t, :, :, :], [h, c]); output_inner.append(h)
#             cur_in = torch.stack(output_inner, dim=1)
#         return cur_in, [h, c]
#     def _init_hidden(self, b, i, d): return [cell.init_hidden(b, i, d) for cell in self.cell_list]

# class ConvLSTMNet(nn.Module):
#     def __init__(self, input_dim, forecast_horizon, hidden_dims=[64, 32], kernel_size=(3, 3)):
#         super(ConvLSTMNet, self).__init__(); self.cl1 = ConvLSTM(input_dim, hidden_dims[0], kernel_size, 1, batch_first=True); self.cl2 = ConvLSTM(hidden_dims[0], hidden_dims[1], kernel_size, 1, batch_first=True); self.output_conv = nn.Conv2d(hidden_dims[1], forecast_horizon, kernel_size=(1, 1), padding='same')
#     def forward(self, x_seq): l1_o, _ = self.cl1(x_seq); l2_o, _ = self.cl2(l1_o); return self.output_conv(l2_o[:, -1, :, :, :])

# def train_model(model, train_loader, val_loader, device, epochs, lr, patience, model_path):
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.MSELoss(); best_val_loss = float('inf'); scaler = torch.cuda.amp.GradScaler(); epochs_no_improve = 0
#     print("\n--- Starting Model Training ---")
#     for epoch in range(epochs):
#         start_time = time.time(); model.train(); total_train_loss = 0.0; train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]")
#         for X, y in train_pbar:
#             X, y = X.to(device), y.to(device); optimizer.zero_grad(set_to_none=True)
#             with torch.cuda.amp.autocast(): predicted = model(X); target = y; loss = loss_fn(predicted, target)
#             scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); total_train_loss += loss.item(); train_pbar.set_postfix({'loss': f'{loss.item():.6f}'})
#         avg_train_loss = total_train_loss / len(train_loader); model.eval(); total_val_loss = 0.0
#         with torch.no_grad():
#             for X, y in val_loader:
#                 X, y = X.to(device), y.to(device)
#                 with torch.cuda.amp.autocast(): predicted = model(X); target = y; loss = loss_fn(predicted, target)
#                 total_val_loss += loss.item()
#         avg_val_loss = total_val_loss / len(val_loader); epoch_time = time.time() - start_time
#         print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}")
#         if avg_val_loss < best_val_loss:
#             best_val_loss = avg_val_loss; epochs_no_improve = 0; torch.save(model.state_dict(), model_path); print(f"✅ New best model saved with validation loss: {best_val_loss:.6f}")
#         else:
#             epochs_no_improve += 1
#             if epochs_no_improve >= patience: print(f"Early stopping triggered after {epoch+1} epochs."); break
#     print(f"\n✅ Training completed. Best model saved to {model_path}"); model.load_state_dict(torch.load(model_path)); return model

In [4]:
# ---------------------------test cell------------------------------------

import xarray as xr
import numpy as np

# Use the same file path from your notebook's configuration cell
STATIC_DATA_PATH = r'c:\Users\user\Documents\GitHub\Wave-Prediction\dAtA\cmems_mod_ibi_wav_my_0.027deg_static_1756186965833.nc'

try:
    # Load the dataset using the xarray library
    ds_static = xr.open_dataset(STATIC_DATA_PATH)

    print("--- Static Dataset Information ---")
    print(ds_static)
    print("\n" + "="*30 + "\n")

    # Xarray datasets store coordinates in the .coords attribute.
    # Common names are 'latitude', 'longitude', 'lat', or 'lon'.
    if 'latitude' in ds_static.coords and 'longitude' in ds_static.coords:
        lats = ds_static.coords['latitude'].values
        lons = ds_static.coords['longitude'].values

        print(f"Successfully found coordinates.")
        print(f"Latitude ranges from {np.min(lats):.4f} to {np.max(lats):.4f}")
        print(f"Longitude ranges from {np.min(lons):.4f} to {np.max(lons):.4f}")

    else:
        print("Could not find coordinates named 'latitude' and 'longitude'.")
        print("Check the dataset information printed above for the correct coordinate names (e.g., 'lat', 'lon').")

except FileNotFoundError:
    print(f"❌ ERROR: The file was not found at the path: {STATIC_DATA_PATH}")
except Exception as e:
    print(f"An error occurred: {e}")

--- Static Dataset Information ---
<xarray.Dataset> Size: 52kB
Dimensions:    (latitude: 72, longitude: 90)
Coordinates:
  * latitude   (latitude) float32 288B 38.5 38.53 38.56 ... 40.42 40.44 40.47
  * longitude  (longitude) float32 360B -11.0 -10.97 -10.94 ... -8.555 -8.527
Data variables:
    mask       (latitude, longitude) float32 26kB ...
    deptho     (latitude, longitude) float32 26kB ...
Attributes:
    Conventions:       CF-1.11
    title:             Static files for product IBI_MULTIYEAR_WAV_005_006
    institution:       Nologin-MeteoFrance
    credit:            E.U. Copernicus Marine Service Information
    contact:           https://marine.copernicus.eu/contact
    references:        http://marine.copernicus.eu
    comment:           
    subset:source:     ARCO data downloaded from the Marine Data Store using ...
    subset:productId:  IBI_MULTIYEAR_WAV_005_006
    subset:datasetId:  cmems_mod_ibi_wav_my_0.027deg_static_202311--ext--bathy
    subset:date:       2025-0

In [14]:
# --- Main Execution Logic ---
if __name__ == '__main__':
    train_dir = os.path.join(PROCESSED_DATA_DIR, 'train')
    val_dir = os.path.join(PROCESSED_DATA_DIR, 'val')
    train_dataset = PreprocessedWaveDataset(train_dir)
    val_dataset = PreprocessedWaveDataset(val_dir)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True if NUM_WORKERS > 0 else False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True if NUM_WORKERS > 0 else False)
    print(f"\nDataLoaders created with {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")
    
    model = ConvLSTMNet(input_dim=INPUT_CHANNELS, forecast_horizon=FORECAST_HORIZON_HOURS)
    model.to(device)
    print(f"Model built with {INPUT_CHANNELS} input channels and a {FORECAST_HORIZON_HOURS}-hour forecast horizon.")

    trained_model = train_model(
        model=model, train_loader=train_loader, val_loader=val_loader, device=device,
        epochs=EPOCHS, lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE, model_path=MODEL_SAVE_PATH
    )


DataLoaders created with 26088 training samples and 4128 validation samples.
Model built with 11 input channels and a 168-hour forecast horizon.

--- Starting Model Training ---


Epoch 1/50 [Training]:   0%|          | 0/6522 [00:00<?, ?it/s]

KeyboardInterrupt: 

# STEP 3: EVALUATION AND VISUALIZATION SCRIPT

In [ ]:
# ==============================================================================
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import os
import pickle
import glob
from torch.utils.data import Dataset, DataLoader

print("--- Starting Final Evaluation on the Test Set ---\n")

# --- Define Classes ---
class PreprocessedWaveDataset(Dataset):
    def __init__(self, split_dir):
        self.file_paths = sorted(glob.glob(os.path.join(split_dir, '*.pt')))
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        return torch.load(self.file_paths[idx])

class ConvLSTMCell(nn.Module):
    def __init__(self, i, h, k, b): super(ConvLSTMCell, self).__init__(); self.input_dim, self.hidden_dim, self.kernel_size, self.bias = i, h, k, b; self.padding = k[0] // 2; self.conv = nn.Conv2d(i + h, 4 * h, k, padding=self.padding, bias=b)
    def forward(self, x, h_c): h, c = h_c; combined = torch.cat([x, h], dim=1); cc = self.conv(combined); i, f, o, g = torch.split(cc, self.hidden_dim, dim=1); i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g); c_n = f * c + i * g; h_n = o * torch.tanh(c_n); return h_n, c_n
    def init_hidden(self, b, i, d): h, w = i; return (torch.zeros(b, self.hidden_dim, h, w, device=d), torch.zeros(b, self.hidden_dim, h, w, device=d))

class ConvLSTM(nn.Module):
    def __init__(self, i, h, k, n, batch_first=True, bias=True):
        super(ConvLSTM, self).__init__(); self.batch_first, self.num_layers = batch_first, n; h_dims = [h] * n if isinstance(h, int) else h; c_list = [];
        for j in range(self.num_layers): c_i_dim = i if j == 0 else h_dims[j - 1]; c_list.append(ConvLSTMCell(c_i_dim, h_dims[j], k, bias)); self.cell_list = nn.ModuleList(c_list)
    def forward(self, x, h_c=None):
        b, s_l, _, h, w = x.size();
        if h_c is None: h_c = self._init_hidden(b, (h, w), x.device)
        cur_in = x
        for l_idx in range(self.num_layers):
            h, c = h_c[l_idx]; out_inner = []
            for t in range(s_l): h, c = self.cell_list[l_idx](cur_in[:, t, :, :, :], [h, c]); out_inner.append(h)
            cur_in = torch.stack(out_inner, dim=1)
        return cur_in, [h, c]
    def _init_hidden(self, b, i, d): return [cell.init_hidden(b, i, d) for cell in self.cell_list]

class ConvLSTMNet(nn.Module):
    def __init__(self, i, f, h_dims=[64, 32], k=(3, 3)): super(ConvLSTMNet, self).__init__(); self.cl1 = ConvLSTM(i, h_dims[0], k, 1, batch_first=True); self.cl2 = ConvLSTM(h_dims[0], h_dims[1], k, 1, batch_first=True); self.output_conv = nn.Conv2d(h_dims[1], f, kernel_size=(1, 1), padding='same')
    def forward(self, x): l1, _ = self.cl1(x); l2, _ = self.cl2(l1); return self.output_conv(l2[:, -1, :, :, :])

# --- Load Model and Prepare Test Loader ---
try:
    with open(os.path.join(PROCESSED_DATA_DIR, 'scalers.pkl'), 'rb') as f: scalers = pickle.load(f)
except FileNotFoundError: raise SystemExit(f"❌ ERROR: Scalers file not found for lookback={LOOKBACK_HOURS}.")

test_dir = os.path.join(PROCESSED_DATA_DIR, 'test')
test_dataset = PreprocessedWaveDataset(test_dir)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model = ConvLSTMNet(input_dim=INPUT_CHANNELS, forecast_horizon=FORECAST_HORIZON_HOURS)
try:
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
except FileNotFoundError: raise SystemExit(f"❌ ERROR: Model file '{MODEL_SAVE_PATH}' not found.")

if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
model.to(device)
model.eval()
print("✅ Best model loaded successfully.")

# --- Evaluation and Visualization Functions ---
def evaluate_and_visualize(model, test_loader, device, scalers, target_var):
    model.eval(); total_rmse_sum_sq, total_mae_sum, total_samples = 0, 0, 0; vis_data = {'predictions': [], 'targets': []}
    print("\n--- Generating predictions for the test set... ---\n")
    with torch.no_grad():
        for i, (X, y) in enumerate(tqdm(test_loader, desc="Evaluating Test Set")):
            X, y = X.to(device), y.to(device); predictions_scaled = model(X); target_scaled = y; target_scaler = scalers[target_var]
            pred_original = target_scaler.inverse_transform(predictions_scaled.cpu().numpy().reshape(-1, 1)).reshape(predictions_scaled.shape)
            target_original = target_scaler.inverse_transform(target_scaled.cpu().numpy().reshape(-1, 1)).reshape(target_scaled.shape)
            total_rmse_sum_sq += np.sum((pred_original - target_original) ** 2); total_mae_sum += np.sum(np.abs(pred_original - target_original)); total_samples += pred_original.size
            if len(vis_data['predictions']) < 2: vis_data['predictions'].append(pred_original[0]); vis_data['targets'].append(target_original[0])
    rmse = np.sqrt(total_rmse_sum_sq / total_samples); mae = total_mae_sum / total_samples
    print(f"\n📊 Test Results (all {FORECAST_HORIZON_HOURS} hours):\n  RMSE: {rmse:.4f}\n  MAE:  {mae:.4f}")
    return vis_data

def plot_visual_comparison(vis_data, save_path, timesteps_to_plot):
    num_timesteps = len(timesteps_to_plot)
    for i in range(len(vis_data['predictions'])):
        fig, axes = plt.subplots(num_timesteps, 3, figsize=(18, 5 * num_timesteps), squeeze=False); plt.suptitle(f"Visual Comparison for Sample {i+1}", fontsize=18, y=0.99)
        for row, t_idx in enumerate(timesteps_to_plot):
            pred_s, targ_s = vis_data['predictions'][i][t_idx, :, :], vis_data['targets'][i][t_idx, :, :]; v_max = max(np.max(pred_s), np.max(targ_s), 0.1)
            im1 = axes[row, 0].imshow(targ_s, cmap='viridis', vmin=0, vmax=v_max); axes[row, 0].set_title(f'Target (Hour {t_idx+1})'); fig.colorbar(im1, ax=axes[row, 0])
            im2 = axes[row, 1].imshow(pred_s, cmap='viridis', vmin=0, vmax=v_max); axes[row, 1].set_title(f'Prediction (Hour {t_idx+1})'); fig.colorbar(im2, ax=axes[row, 1])
            diff = pred_s - targ_s; diff_max = np.max(np.abs(diff)) if np.max(np.abs(diff)) > 0 else 0.1; im3 = axes[row, 2].imshow(diff, cmap='RdBu_r', vmin=-diff_max, vmax=diff_max); axes[row, 2].set_title(f'Error (Hour {t_idx+1})'); fig.colorbar(im3, ax=axes[row, 2])
        plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.savefig(f"{save_path}_{i}.png", dpi=200); plt.show()
    print(f"✅ Comparison maps saved to '{save_path}_X.png'")

# --- Run Evaluation and Generate Visualizations ---
vis_data = evaluate_and_visualize(model, test_loader, device, scalers, TARGET_VAR)

print("\n--- Generating Visualization 1: Comparison Maps ---")
plot_visual_comparison(vis_data, save_path=f'test_comparison_lookback_{LOOKBACK_HOURS}', timesteps_to_plot=[0, 23, 71, 167])

print("\n--- Generating Visualization 2 & 3: Scatter and Error Plots ---")
all_preds_flat = np.array(vis_data['predictions']).flatten(); all_targets_flat = np.array(vis_data['targets']).flatten(); errors = all_preds_flat - all_targets_flat
plt.figure(figsize=(8, 8)); sample_indices = np.random.choice(len(all_preds_flat), min(len(all_preds_flat), 10000), replace=False)
plt.scatter(all_targets_flat[sample_indices], all_preds_flat[sample_indices], alpha=0.3, s=10); min_val, max_val = min(all_targets_flat.min(), all_preds_flat.min()), max(all_targets_flat.max(), all_preds_flat.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction'); plt.xlabel("Actual Wave Height (m)"); plt.ylabel("Predicted Wave Height (m)"); plt.title(f"Scatter Plot (Lookback: {LOOKBACK_HOURS}hrs)"); plt.grid(True); plt.legend(); plt.axis('equal'); plt.tight_layout()
plt.savefig(f'scatter_plot_lookback_{LOOKBACK_HOURS}.png', dpi=300); plt.show(); print(f"✅ Scatter plot saved.")

mean_error = np.mean(errors); plt.figure(figsize=(10, 6)); plt.hist(errors, bins=100, density=True)
plt.axvline(mean_error, color='r', linestyle='--', lw=2, label=f'Mean Error: {mean_error:.3f}'); plt.title(f"Distribution of Prediction Errors (Lookback: {LOOKBACK_HOURS}hrs)"); plt.xlabel("Error (Predicted - Actual) in meters"); plt.ylabel("Density")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(f'error_histogram_lookback_{LOOKBACK_HOURS}.png', dpi=300); plt.show(); print(f"✅ Error histogram saved.")

print("\n✅ All evaluation visualizations are complete.")